# Demand Forecasting — Exploration Notebook

This notebook walks through the essence of the **02-demand-forecasting** case study:
a hierarchical daily-unit demand forecast (4 stores × 6 products) whose models must
survive an embargoed expanding-window backtest, not just beat a random split.

It demonstrates three things, all from **persisted artifacts** (no downloads, no
network):

1. Loading the prepared panel, feature frame and dataset identity card via the
   package's own settings/config.
2. The headline 6-run comparison ladder straight from `reports/metrics.json`.
3. Reproducing one small key result out-of-sample: `naive_last` (persistence)
   vs the production `hgb_tuned` model on the last backtest fold, using the
   same leakage-safe evaluation helpers the pipeline uses (~20 s).

## How to run

From the repository root:

```bash
uv sync --all-packages
```

Then run the cells from the **project directory** (`projects/02-demand-forecasting`),
either with `jupyter lab`/`vscode` or by converting to a script:

```bash
cd projects/02-demand-forecasting
uv run jupyter nbconvert --to notebook --execute --inplace notebooks/exploration.ipynb
```

All data is read from `data/` (prepared parquet + DVC-tracked artifacts), produced
by `uv run dvc repro` — nothing here re-downloads or regenerates the dataset.

## 1. Load the prepared data

Paths come from the package's `DemandSettings` (ADR-0002: env-driven settings),
so this resolves to `data/` relative to the project directory. The dataset
identity card (`data/dataset.json`) is written by the `prepare` DVC stage and
carries the SHA-256 checksum and DQ-gate metadata.

In [ ]:
import json

import polars as pl
from demand_forecasting.config import DemandSettings, ForecastConfig

settings = DemandSettings()  # data_dir=data, reports_dir=reports (relative to cwd)
config = ForecastConfig()  # forecast problem definition (730 days, 4x6, lags [1,7,14])

identity = json.loads((settings.data_dir / "dataset.json").read_text())
panel = pl.read_parquet(settings.data_dir / "panel.parquet")
features = pl.read_parquet(settings.data_dir / "features.parquet")

print(f"dataset: {identity['name']} v{identity['version']} ({identity['source']})")
print(f"checksum sha256: {identity['checksum_sha256'][:16]}...  n_rows: {identity['n_rows']}")
print(
    f"panel: {panel.height} rows, {panel['date'].n_unique()} dates, "
    f"{panel['store'].n_unique()} stores x {panel['product'].n_unique()} products"
)
print(f"features: {features.height} dated rows, cols: {features.columns}")

## 2. Headline comparison — the run ladder

`reports/metrics.json` holds the pooled out-of-fold metrics (MAE / RMSE / MAPE)
for every run that passed through the embargoed expanding-window backtest —
including the naives, which are the floor every real model must beat. The key
number: the tuned GBM lands **-41% MAE vs persistence**.

In [ ]:
metrics = json.loads((settings.reports_dir / "metrics.json").read_text())

naive_last_mae = metrics["naive_last"]["mae"]
rows = [
    {
        "run": name,
        "MAE": round(m["mae"], 3),
        "RMSE": round(m["rmse"], 3),
        "MAPE %": round(m["mape_pct"], 1),
        "dMAE vs naive_last": "(floor)"
        if name == "naive_last"
        else f"{(m['mae'] / naive_last_mae - 1):+.1%}",
    }
    for name, m in metrics.items()
]
print(pl.DataFrame(rows))

## 3. Reproduce one key result — naive vs model, out-of-sample

Re-run the last backtest fold (fold 4 of 4, 7-day embargo) for just two runs —
the strongest naive (`naive_last`, per-series persistence) and the production
`hgb_tuned` GBM — using the package's own `cv_row_folds` / `to_xy` / `build`
helpers. This is the same protocol as the full pipeline (`cli evaluate`), just
restricted to one fold and two models, so it finishes in seconds.

Expect `hgb_tuned` MAE ≈ 9.5 vs `naive_last` ≈ 16.2 on this fold — the same
direction and rough magnitude as the pooled ladder above.

In [ ]:
import numpy as np
from demand_forecasting.evaluate import cv_row_folds
from demand_forecasting.model import model_specs, to_xy

folds = cv_row_folds(features, config)
specs = {s.name: s for s in model_specs()}
train_rows, val_rows = folds[-1]  # last expanding window

last_fold_mae = {}
for name in ("naive_last", "hgb_tuned"):
    spec = specs[name]
    x, y, dates, series = to_xy(features, config, spec)
    model = spec.build(config)
    model.fit(x[train_rows], y[train_rows])
    y_pred = model.predict(x[val_rows])
    last_fold_mae[name] = float(np.mean(np.abs(y[val_rows] - y_pred)))

reduction = 1.0 - last_fold_mae["hgb_tuned"] / last_fold_mae["naive_last"]
print(
    f"last fold: {len(val_rows)} out-of-fold rows over "
    f"{len(np.unique(dates[val_rows]))} dates x 24 series"
)
print(f"naive_last MAE = {last_fold_mae['naive_last']:.3f}")
print(
    f"hgb_tuned  MAE = {last_fold_mae['hgb_tuned']:.3f}  "
    f"({-reduction:+.1%} MAE vs persistence, out-of-sample)"
)

## 4. Two more persisted artifacts, read honestly

The pipeline also ships reconciliation and probabilistic-band results. Read
them as-is — including the imperfect parts, which the case study reports
rather than hides.

In [ ]:
recon = json.loads((settings.reports_dir / "reconciliation.json").read_text())
prob = json.loads((settings.reports_dir / "probabilistic.json").read_text())

print(f"reconciliation (MinT-lite, evaluated on embargoed fold {recon['eval_fold'] + 1}):")
print(f"  chosen shrinkage w           = {recon['shrinkage']}")
print(f"  store MAE  {recon['store_mae_before']:.3f} -> {recon['store_mae_after']:.3f}")
print(f"  incoherence {recon['incoherence_before']:.3f} -> {recon['incoherence_after']:.3f}")
print(f"\nprobabilistic bands (target coverage {prob['target_coverage']:.0%}):")
print(
    f"  quantile band coverage (pooled) = "
    f"{prob['quantile_band']['coverage_pooled']:.3f}  <- under-covers, reported as-is"
)
print(
    f"  split-conformal coverage (last fold) = {prob['split_conformal']['coverage_last_fold']:.3f}"
)

## Where to go next

- Case study (full narrative, figures, results tables): `/projects/02-demand-forecasting/`
- Design record (architecture decisions, why global models + one-hot identity,
  why embargoed time-CV): `DESIGN.md`
- Code tree: `src/demand_forecasting/` (`data.py` panel + DQ gates,
  `features.py` shift-only lags, `model.py` rungs, `evaluate.py` backtest,
  `hpo.py`, `probabilistic.py`, `reconcile.py`, `cli.py`)
- Reproduce everything end-to-end: `uv run dvc repro` from the project directory.